# 一、基于ChromaDB的向量数据库实践

#### 0. 安装依赖

In [1]:
%pip install chromadb sentence-transformers

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/21.4 MB ? eta -:--:--
      --------------------------------------- 0.5/21.4 MB 2.8 MB/s eta 0:00:08
     -- ------------------------------------- 1.6/21.4 MB 4.4 MB/s eta 0:00:05
     ---- ----------------------------------- 2.4/21.4 MB 4.2 MB/s eta 0:00:05
     ------ --------------------------------- 3.4/21.4 MB 4.3 MB/s eta 0:00:05
     -------- ------------------------------- 4.5/21.4 MB 4.5 MB/s eta 0:00:04
     ---------- ----------------------------- 5.8/21.4 MB 4.7 MB/s eta 0:00:04
     ------------ --------------------------- 6.8/21.4 MB 4.8 MB/s eta 0:00:04
     --------------- ------------------------ 8.4/21.4 MB 5.1 MB/s eta 0:00:03
     ----------------- ---------------------- 9.4/21.4 MB 5.1 MB/s eta 0:00:03
     -------------------- ------------------- 11.0/21.4 MB 5.3 MB/s eta 0:00:02
     ----------------------- ---------------- 12.6/21.4 MB 5.5 MB/s eta 0:00

#### 1. 初始化 ChromaDB 客户端：使用 chromadb.Client() 来连接并创建数据库。

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
from typing import List

# 初始化 Chroma 客户端
client = chromadb.Client()

# 加载预训练的 Sentence Transformer 模型
model = SentenceTransformer('./all-MiniLM-L6-v2')

d:\miniconda\envs\py310\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
d:\miniconda\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 创建 Chroma Collection（数据库）
collection = client.create_collection('my_documents')

#### 2. 向量嵌入：使用 SentenceTransformer 加载预训练的模型（如 all-MiniLM-L6-v2），并将文档转换为嵌入向量。

In [4]:
def add_documents_to_chroma(documents: List[str]):
    """
    将文档添加到 Chroma 数据库中
    """
    # 获取文档的嵌入
    embeddings = model.encode(documents)
    
    # 将文档和它们的嵌入存入数据库
    collection.add(
        documents=documents,  # 文档
        embeddings=embeddings,  # 嵌入向量
        metadatas=[{'source': f"doc_{i}"} for i in range(len(documents))],  # 元数据
        ids=[str(i) for i in range(len(documents))]  # 文档 ID
    )

#### 3. 添加文档到 Chroma：将文档及其嵌入存入 Chroma 的 Collection 中。

In [5]:
def query_chroma(query: str, top_k: int = 1):
    """
    执行查询并返回相似的文档
    """
    # 获取查询的嵌入
    query_embedding = model.encode([query])
    
    # 使用 Chroma 进行相似度检索
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k  # 返回最相似的 top_k 个文档
    )
    
    return results

#### 4. 查询与检索：通过输入查询文本，将查询文本转换为嵌入向量，并在数据库中查找与查询最相似的文档。

In [6]:
def load_txt_documents(root_dir):
    """
    遍历 root_dir 下的所有 .txt 文件，读取内容并返回 documents 列表。
    
    参数:
        root_dir (Path or str): 根目录路径（支持 pathlib.Path）
    
    返回:
        documents (list[str]): 所有成功读取的文档文本内容列表
        labels (list[str]): 对应的类别标签（上一级目录名）
        file_paths (list[str]): 对应的文件路径列表
    """
    from pathlib import Path
    
    root_dir = Path(root_dir)
    documents = []
    labels = []
    file_paths = []
    
    # 遍历所有 txt 文件
    for txt_path in root_dir.rglob("*.txt"):
        # 类别用上一级目录名表示，比如 C3-Art
        label = txt_path.parent.name
        
        # 读取文件内容，兜底多种编码
        text = None
        for encoding in ["utf-8", "gb18030", "gbk"]:
            try:
                with open(txt_path, "r", encoding=encoding) as f:
                    text = f.read().strip()
                break
            except UnicodeDecodeError:
                continue
        
        if not text:
            # print(f"跳过无法解码的文件: {txt_path}")
            continue
        
        if len(text) == 0:
            continue  # 空文件跳过
        
        documents.append(text)
        labels.append(label)
        file_paths.append(str(txt_path))
    
    # print(f"共读取到 {len(documents)} 篇文档。")
    # 如果需要查看示例，可取消下方注释
    # if documents:
    #     print("示例文档内容：")
    #     print(documents[0][:200])
    #     print("对应类别：", labels[0])
    #     print("对应文件：", file_paths[0])
    
    return documents, labels, file_paths

In [7]:
import os
from pathlib import Path

root_dir = Path("./train/C3-Art")
documents, labels, file_paths = load_txt_documents(root_dir)

# 如果只需要 documents：
# documents, _, _ = load_txt_documents(root_dir)

# 继续添加至 Chroma
add_documents_to_chroma(documents)

In [8]:
# 执行查询
query = "什么是“形神俱佳”？"
results = query_chroma(query)

# 输出查询结果
print("查询结果:")
for result in results['documents']:
    print(result)

查询结果:
['【 文献号 】2-535\n【原文出处】文艺报\n【原刊地名】京\n【原刊期号】19990624\n【原刊页号】②\n【分 类 号】J1\n【分 类 名】文艺理论\n【复印期号】199908\n【 标  题 】铁肩担道义\n    ——文艺工作者的精神价值取向\n【 作  者 】陆贵山\n【编 者 按】九十年代以来，文学界相继参与了“人文精神”、“知识分子话语”等话题的讨论，这些话题都涉及到文艺的精神价值取向。在市\n场经济的冲击下，当前有一种忽略文艺的精神价值取向的倾向。谈论这个话题时，我们不可忽视作家、评论家主体取向的关键作用，而主体价值\n取向的多样选择则是客观存在的，这也决定了探讨文艺的精神价值时必然出现的歧义；然而有一点却是我们应当认同的，即“文艺是国民精神所\n发的火光，同时也是引导国民精神的前途的灯火。”（鲁迅语）面对忽略或漠视文艺精神价值取向的现象，重提这个话题对于文艺创作是有其现\n实意义的，也有益于对九十年代以来思想文化界的反思作一深化和总结。\n    我们特开辟关于文艺的精神价值取向的讨论专栏，以陆贵山的这篇文章作为开头，陆续还有几位理论家从不同的角度各抒己见，他们之间的\n见解不尽然相同，甚至可能是相左的，但这并不妨碍我们进行充分说理的、富有理论建设的争鸣和探讨。\n【 正  文 】\n            一、政治良知\n    曾经发生的政治动荡和政治专制所带来的沉重的创伤象梦幻一样缠绕着人们的头脑。这种可怕的政治运动政治风暴造成了十分严重的政治恶\n果，很大程度上引发和酿成了作家和群众的政治神经的脆弱和麻痹，政治意识的模糊和退化，政治良知和政治责任的消解和隐匿。一种厌恶和漠\n视政治的非理性和情绪化的心态不可遏制地弥漫开来。然而，政治是具有不同性质的。反对专制的非人的政治，应当是为了追求和守护人民利益\n的民主的政治。世界尚未进入大同，阳光下面还有罪恶。当南斯拉夫的神圣领土和善良人民遭受到邪恶势力的狂轰滥炸的时候，南斯拉夫的文艺\n工作者们通过举办音乐会的形式，发出怒吼和呐喊，向侵略势力表示义愤和抗争、声讨和控诉。这不禁使人们想起了难忘的战争岁月，唤起了已\n经被淡忘了的民族生存危机的历史境况下文艺所负有的政治使命的记忆。即便是和平发展的时代，也往往笼罩着战争的风云，霸权主义者总会伸\n出侵略的魔爪。强权政治已经成为

# 二、基于 FAISS 向量数据库的代码示例

In [9]:
%pip install faiss-cpu sentence-transformers

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/
     ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
     - -------------------------------------- 0.5/18.9 MB 8.5 MB/s eta 0:00:03
     --- ------------------------------------ 1.8/18.9 MB 7.7 MB/s eta 0:00:03
     ------- -------------------------------- 3.4/18.9 MB 6.9 MB/s eta 0:00:03
     ---------- ----------------------------- 5.0/18.9 MB 7.2 MB/s eta 0:00:02
     ------------- -------------------------- 6.3/18.9 MB 7.0 MB/s eta 0:00:02
     ---------------- ----------------------- 7.6/18.9 MB 6.9 MB/s eta 0:00:02
     ------------------ --------------------- 8.9/18.9 MB 6.8 MB/s eta 0:00:02
     ---------------------- ----------------- 10.5/18.9 MB 7.1 MB/s eta 0:00:02
     -------------------------- ------------- 12.3/18.9 MB 7.1 MB/s eta 0:00:01
     ----------------------------- ---------- 14.2/18.9 MB 7.3 MB/s eta 0:00:01
     ----------------------------- ---------- 14.2/18.9 MB 7.3 MB/s eta 0

#### 1. FAISS 索引创建：使用 faiss.IndexFlatL2(dim) 创建一个基于 L2 距离的平面索引。

In [10]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List

# 加载预训练的 Sentence Transformer 模型
model = SentenceTransformer('./all-MiniLM-L6-v2')

# 创建一个 FAISS 索引
dim = 384  # MiniLM-L6-v2 输出向量的维度
index = faiss.IndexFlatL2(dim)  # 使用 L2 距离度量

#### 2. 向量嵌入：使用 SentenceTransformer 加载预训练的模型，将文档转换为嵌入向量。

In [11]:
def add_documents_to_faiss(documents: List[str]):
    """
    将文档添加到 FAISS 向量数据库
    """
    # 获取文档的嵌入
    embeddings = model.encode(documents)
    
    # 将嵌入向量添加到 FAISS 索引中
    embeddings = np.array(embeddings).astype('float32')  # FAISS 要求嵌入是 float32 类型
    index.add(embeddings)

In [12]:
def query_faiss(query: str, top_k: int = 3):
    """
    执行查询并返回相似的文档
    """
    # 获取查询的嵌入
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')

    # 使用 FAISS 进行相似度检索
    distances, indices = index.search(query_embedding, top_k)

    return distances, indices

#### 3. 向量添加到 FAISS 索引：将嵌入向量存入 FAISS 索引。

In [13]:
# 添加文档到 FAISS 向量数据库
# documents = [
#     "向量数据库是存储和检索向量的工具，广泛应用于搜索引擎中。",
#     "FAISS 是 Facebook 开发的一个高效的向量搜索库。",
#     "Chroma 是一个开源的向量数据库，适合存储高维向量并进行快速检索。"
# ]
import os
from pathlib import Path

root_dir = Path("./train/C3-Art")
documents, labels, file_paths = load_txt_documents(root_dir)

add_documents_to_faiss(documents)

#### 4. 查询与检索：将查询转换为嵌入向量，使用 FAISS 的 search 方法查询最相似的文档。

In [14]:
# 执行查询
query = "什么是“形神俱佳”？"
distances, indices = query_faiss(query)

# 输出查询结果
print("查询结果:")
for i, idx in enumerate(indices[0]):
    print(f"文档 ID: {idx}, 距离: {distances[0][i]}, 文本: {documents[idx]}")

查询结果:
文档 ID: 307, 距离: 0.7201513648033142, 文本: 【 文献号 】2-535
【原文出处】文艺报
【原刊地名】京
【原刊期号】19990624
【原刊页号】②
【分 类 号】J1
【分 类 名】文艺理论
【复印期号】199908
【 标  题 】铁肩担道义
    ——文艺工作者的精神价值取向
【 作  者 】陆贵山
【编 者 按】九十年代以来，文学界相继参与了“人文精神”、“知识分子话语”等话题的讨论，这些话题都涉及到文艺的精神价值取向。在市
场经济的冲击下，当前有一种忽略文艺的精神价值取向的倾向。谈论这个话题时，我们不可忽视作家、评论家主体取向的关键作用，而主体价值
取向的多样选择则是客观存在的，这也决定了探讨文艺的精神价值时必然出现的歧义；然而有一点却是我们应当认同的，即“文艺是国民精神所
发的火光，同时也是引导国民精神的前途的灯火。”（鲁迅语）面对忽略或漠视文艺精神价值取向的现象，重提这个话题对于文艺创作是有其现
实意义的，也有益于对九十年代以来思想文化界的反思作一深化和总结。
    我们特开辟关于文艺的精神价值取向的讨论专栏，以陆贵山的这篇文章作为开头，陆续还有几位理论家从不同的角度各抒己见，他们之间的
见解不尽然相同，甚至可能是相左的，但这并不妨碍我们进行充分说理的、富有理论建设的争鸣和探讨。
【 正  文 】
            一、政治良知
    曾经发生的政治动荡和政治专制所带来的沉重的创伤象梦幻一样缠绕着人们的头脑。这种可怕的政治运动政治风暴造成了十分严重的政治恶
果，很大程度上引发和酿成了作家和群众的政治神经的脆弱和麻痹，政治意识的模糊和退化，政治良知和政治责任的消解和隐匿。一种厌恶和漠
视政治的非理性和情绪化的心态不可遏制地弥漫开来。然而，政治是具有不同性质的。反对专制的非人的政治，应当是为了追求和守护人民利益
的民主的政治。世界尚未进入大同，阳光下面还有罪恶。当南斯拉夫的神圣领土和善良人民遭受到邪恶势力的狂轰滥炸的时候，南斯拉夫的文艺
工作者们通过举办音乐会的形式，发出怒吼和呐喊，向侵略势力表示义愤和抗争、声讨和控诉。这不禁使人们想起了难忘的战争岁月，唤起了已
经被淡忘了的民族生存危机的历史境况下文艺所负有的政治使命的记忆。即便是和平发展的时代，也往往笼罩着战争的风云，霸权主义者总会伸
出侵略

# 三、优化向量检索（选做）：多级索引与分片

### 1. 实验目标：

学习如何通过 多级索引 或 分片（sharding） 技术来优化大规模向量检索系统的性能。对于大量数据，单一的向量索引可能无法承载查询负载，因此我们可以使用多级索引或将数据分片存储。

### 2. 任务：

#### 更换不同 embedding 模型并比较检索效果

#### 为向量库添加文档元数据（metadata）并实现基于标签的检索

#### 可视化向量空间（如使用 t-SNE 或 PCA）

#### 将向量检索集成到一个简单的问答系统（RAG 原型）